## Data sources

**UNDP Human Development Index (HDI)**
- Source: https://hdr.undp.org/data-center/documentation-and-downloads
- Release: HDR 2025 (HDR25)
- Reference year used for HDI: 2023

**World Bank World Development Indicators (WDI)**
- Source: https://databank.worldbank.org/source/world-development-indicators
- Indicator codes: NY.GDP.MKTP.KD.ZG, FP.CPI.TOTL.ZG, SL.UEM.TOTL.ZS, SP.DYN.IMRT.IN, EG.ELC.ACCS.ZS, IT.NET.USER.ZS, NV.AGR.TOTL.ZS, NV.IND.TOTL.ZS, NE.TRD.GNFS.ZS
- Years downloaded: 2021-2023 (2023 is the reference year, 2021-2022 kept as a look-back window for missing values)
- Access date: 2026-11-23

**World Bank Country API** — country metadata / region
- Source: https://api.worldbank.org/v2/country
- Access date: 2026-11-23

In [1]:
import pandas as pd

In [2]:
# HDI table, was getting encoding error, so added encoding="cp1252"
hdi = pd.read_csv("HDR25_Composite_indices_complete_time_series.csv", encoding="cp1252")
#hdi.head()

In [3]:
# WDI indicators
wdi = pd.read_csv("WDI_Data_2021_2023.csv", na_values="..")
#wdi.head()

In [ ]:
# World Bank country API, coded from documentation of this API
import requests
resp = requests.get("https://api.worldbank.org/v2/country", params={"format": "json", "per_page": 300})
country_meta = pd.json_normalize(resp.json()[1])
#print(type(country_meta))
country_meta = country_meta[country_meta["region.value"] != "Aggregates"]
#excluded aggregates
country_meta = country_meta[["id", "iso2Code", "name", "region.value", "incomeLevel.value"]]
country_meta.columns = ["iso3", "iso2", "country", "region", "income_level"]
#country_meta.head()

In [5]:
import numpy as np

# Step 4: merge HDI and WDI on ISO3, inspect unmatched countries first
is_aggregate = hdi["iso3"].str.startswith("ZZ")  # ZZ.. codes are regions/aggregates by UNDP
hdi_real = hdi.loc[~is_aggregate]

hdi_lookup = pd.Series(dict(zip(hdi_real["iso3"], hdi_real["hdi_2023"])))
country_name_lookup = pd.Series(dict(zip(hdi_real["iso3"], hdi_real["country"])))

wdi_clean = wdi.dropna(subset=["Country Code"]).copy()
wdi_clean["hdi_2023"] = wdi_clean["Country Code"].map(hdi_lookup)  # this is the merge

wdi_only = sorted(wdi_clean.loc[wdi_clean["hdi_2023"].isnull(), "Country Code"].unique())
hdi_only = sorted(set(hdi_lookup.index) - set(wdi_clean["Country Code"]))
print("WDI countries with no HDI match:", wdi_only)
print("HDI countries with no WDI match:", hdi_only)

WDI countries with no HDI match: ['ABW', 'ASM', 'BMU', 'CHI', 'CUW', 'CYM', 'FRO', 'GIB', 'GRL', 'GUM', 'IMN', 'MAC', 'MAF', 'MCO', 'MNP', 'NCL', 'PRI', 'PRK', 'PYF', 'SXM', 'TCA', 'VGB', 'VIR', 'XKX']
HDI countries with no WDI match: []


In [ ]:
# no actual ISO3 mismatches found above (wdi_only are small territories UNDP doesn't publish HDI for, hdi_only is empty)
# so dropped the unmatched rows and keep going
wdi_clean = wdi_clean[wdi_clean["hdi_2023"].notnull()].copy()

In [ ]:
# collapsed each indicator to one value per country
# use 2023 if available, else fall back to 2022, then 2021 (never later than the HDI reference year), record which year was used
year_cols = ["2023 [YR2023]", "2022 [YR2022]", "2021 [YR2021]"]
year_lookup = {"2023 [YR2023]": 2023, "2022 [YR2022]": 2022, "2021 [YR2021]": 2021}

wdi_clean["value"] = wdi_clean[year_cols[0]]
wdi_clean["year_used"] = year_lookup[year_cols[0]]
for col in year_cols[1:]:
    still_missing = wdi_clean["value"].isnull()
    wdi_clean.loc[still_missing, "value"] = wdi_clean.loc[still_missing, col]
    wdi_clean.loc[still_missing, "year_used"] = year_lookup[col]

In [8]:
# one row per country, one column per indicator (plus a _year column recording which year got used)
indicator_codes = ["NY.GDP.MKTP.KD.ZG", "FP.CPI.TOTL.ZG", "SL.UEM.TOTL.ZS", "SP.DYN.IMRT.IN",
                    "EG.ELC.ACCS.ZS", "IT.NET.USER.ZS", "NV.AGR.TOTL.ZS", "NV.IND.TOTL.ZS", "NE.TRD.GNFS.ZS"]

def indicator_series(code):
    subset = wdi_clean[wdi_clean["Series Code"] == code]
    value_series = pd.Series(dict(zip(subset["Country Code"], subset["value"])))
    year_series = pd.Series(dict(zip(subset["Country Code"], subset["year_used"])))
    return value_series, year_series

combined = {}
for code in indicator_codes:
    combined[code], combined[code + "_year"] = indicator_series(code)

combined["hdi_2023"] = pd.Series(dict(zip(wdi_clean["Country Code"], wdi_clean["hdi_2023"])))
combined["country"] = pd.Series(dict(zip(wdi_clean["Country Code"], wdi_clean["Country Name"])))

dataset = pd.DataFrame(combined)
dataset.shape

(193, 20)

In [ ]:
# Step 6: target y from HDI (hdi_2023 only, not used as a predictor later)
dataset["y"] = np.where(dataset["hdi_2023"] < 0.550, 0, np.where(dataset["hdi_2023"] < 0.800, 1, 2))
dataset.loc[dataset["hdi_2023"].isnull(), "y"] = np.nan  # np.where treats NaN<x as False, fix that here
dataset["y"].value_counts(dropna=False)

y
1.0    93
2.0    74
0.0    26
Name: count, dtype: int64

In [ ]:
# Step 6: DominantSector finding from agriculture/industry shares
agri = dataset["NV.AGR.TOTL.ZS"]
ind = dataset["NV.IND.TOTL.ZS"]
services = 100 - agri - ind
valid = (agri.notnull() & ind.notnull()
         & (agri >= 0) & (agri <= 100) & (ind >= 0) & (ind <= 100)
         & (services >= 0) & (services <= 100))

dataset["DominantSector"] = None  # object dtype - init with np.nan would make it float64 and break the string assigns below
dataset.loc[valid & (agri >= ind) & (agri >= services), "DominantSector"] = "Agriculture"
dataset.loc[valid & (ind >= agri) & (ind >= services), "DominantSector"] = "Industry"
dataset.loc[valid & (services >= agri) & (services >= ind), "DominantSector"] = "Services"
dataset["DominantSector"].value_counts(dropna=False)

DominantSector
Services       176
Industry         8
None             7
Agriculture      2
Name: count, dtype: int64

In [11]:
# Step 7: final report
n_countries = len(dataset)
countries_lost_no_hdi = len(wdi_only)  # WDI countries dropped in Step 4 for having no HDI match

print("Final number of countries:", n_countries)
print("Countries lost (no HDI label available):", countries_lost_no_hdi)
print()
print("Class frequencies (y):")
print(dataset["y"].value_counts(dropna=False))
print()
print("Missingness rate per predictor:")
dataset[indicator_codes].isnull().mean()
# predictor NaNs are left as-is on purpose - not imputed here, that's for the imputation exercise

Final number of countries: 193
Countries lost (no HDI label available): 24

Class frequencies (y):
y
1.0    93
2.0    74
0.0    26
Name: count, dtype: int64

Missingness rate per predictor:


NY.GDP.MKTP.KD.ZG    0.020725
FP.CPI.TOTL.ZG       0.077720
SL.UEM.TOTL.ZS       0.072539
SP.DYN.IMRT.IN       0.010363
EG.ELC.ACCS.ZS       0.000000
IT.NET.USER.ZS       0.056995
NV.AGR.TOTL.ZS       0.036269
NV.IND.TOTL.ZS       0.036269
NE.TRD.GNFS.ZS       0.119171
dtype: float64

In [ ]:
# Step 8: writing the fixed dataset, these codes used from documentation of library pandas
dataset.index.name = "country_code"
final = dataset.reset_index()
final.to_csv("DS203_final_dataset.csv", index=False)
final.shape

(193, 23)

In [ ]:
# Step 8: data dictionary (column, description, source, year-selection rule, manual corrections)
indicator_descriptions = {
    "NY.GDP.MKTP.KD.ZG": "GDP growth (annual %)",
    "FP.CPI.TOTL.ZG": "Inflation, consumer prices (annual %)",
    "SL.UEM.TOTL.ZS": "Unemployment, total (% of labor force)",
    "SP.DYN.IMRT.IN": "Mortality rate, infant (per 1,000 live births)",
    "EG.ELC.ACCS.ZS": "Access to electricity (% of population)",
    "IT.NET.USER.ZS": "Individuals using the Internet (% of population)",
    "NV.AGR.TOTL.ZS": "Agriculture, value added (% of GDP)",
    "NV.IND.TOTL.ZS": "Industry, value added (% of GDP)",
    "NE.TRD.GNFS.ZS": "Trade (% of GDP)",
}

column, description, source, year_rule, manual_correction = [], [], [], [], []
for code in indicator_codes:
    column.append(code); description.append(indicator_descriptions[code])
    source.append("World Bank WDI, https://databank.worldbank.org/source/world-development-indicators")
    year_rule.append("2023 if present, else 2022, else 2021"); manual_correction.append("none")
    column.append(code + "_year"); description.append("year actually used for " + code)
    source.append("derived"); year_rule.append("see " + code); manual_correction.append("none")

column += ["hdi_2023", "country", "y", "DominantSector"]
description += [
    "UNDP Human Development Index score, 2023",
    "country name (from WDI)",
    "target class: 0=Low (<0.550), 1=Medium/High (0.550-0.800), 2=Very High (>=0.800)",
    "largest of agriculture/industry/services share of GDP",
]
source += [
    "UNDP HDR25, https://hdr.undp.org/data-center/documentation-and-downloads",
    "World Bank WDI",
    "derived from hdi_2023",
    "derived from NV.AGR.TOTL.ZS, NV.IND.TOTL.ZS",
]
year_rule += ["2023 only", "n/a", "n/a", "n/a"]
manual_correction += ["none", "none", "none", "none"]

data_dictionary = pd.DataFrame({
    "column": column,
    "description": description,
    "source": source,
    "year_selection_rule": year_rule,
    "manual_correction": manual_correction,
})
data_dictionary.to_csv("DS203_data_dictionary.csv", index=False)
data_dictionary

,column,description,source,year_selection_rule,manual_correction
0,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),"World Bank WDI, https://databank.worldbank.org...","2023 if present, else 2022, else 2021",none
1,NY.GDP.MKTP.KD.ZG_year,year actually used for NY.GDP.MKTP.KD.ZG,derived,see NY.GDP.MKTP.KD.ZG,none
2,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)","World Bank WDI, https://databank.worldbank.org...","2023 if present, else 2022, else 2021",none
3,FP.CPI.TOTL.ZG_year,year actually used for FP.CPI.TOTL.ZG,derived,see FP.CPI.TOTL.ZG,none
4,SL.UEM.TOTL.ZS,"Unemployment, total (% of labor force)","World Bank WDI, https://databank.worldbank.org...","2023 if present, else 2022, else 2021",none
5,SL.UEM.TOTL.ZS_year,year actually used for SL.UEM.TOTL.ZS,derived,see SL.UEM.TOTL.ZS,none
6,SP.DYN.IMRT.IN,"Mortality rate, infant (per 1,000 live births)","World Bank WDI, https://databank.worldbank.org...","2023 if present, else 2022, else 2021",none
7,SP.DYN.IMRT.IN_year,year actually used for SP.DYN.IMRT.IN,derived,see SP.DYN.IMRT.IN,none
8,EG.ELC.ACCS.ZS,Access to electricity (% of population),"World Bank WDI, https://databank.worldbank.org...","2023 if present, else 2022, else 2021",none
9,EG.ELC.ACCS.ZS_year,year actually used for EG.ELC.ACCS.ZS,derived,see EG.ELC.ACCS.ZS,none
